In [1]:
import torch
import scanpy as sc
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm

# ==========================================
# 路径配置 (请务必修改 RAW_DATA_DIR)
# ==========================================
# 1. 训练好的特征路径 (不变)
EMB_DATA_DIR = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/kz_data/Graph_embeddings/"

# 2. 【关键】原始数据路径 (这里存放着带坐标的 .h5ad 或 .csv)
# 假设你的原始数据也是 .h5ad 格式，并且文件名和 .pt 能对应上
RAW_DATA_DIR = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/kz_data/Raw_Data/Log1p/"
# 如果你的原始数据在其他地方，请修改上面这一行！

# 3. 结果保存路径
RESULT_DIR = "spatial_domains_result"
if not os.path.exists(RESULT_DIR):
    os.makedirs(RESULT_DIR)

def load_coords_from_original(filename_stem):
    """
    根据文件名，去 RAW_DATA_DIR 找对应的原始文件读取坐标
    filename_stem: 不带后缀的文件名，如 'ColorectalCancer_10x'
    """
    # 假设原始文件是 .h5ad 格式
    h5ad_path = os.path.join(RAW_DATA_DIR, f"{filename_stem}.h5ad")
    
    if os.path.exists(h5ad_path):
        try:
            adata_orig = sc.read_h5ad(h5ad_path)
            # 优先找 obsm['spatial']
            if 'spatial' in adata_orig.obsm:
                return adata_orig.obsm['spatial']
            # 有些数据存在 obs['x'] 和 obs['y'] 里
            elif 'x' in adata_orig.obs and 'y' in adata_orig.obs:
                return np.column_stack((adata_orig.obs['x'], adata_orig.obs['y']))
            else:
                print(f"⚠️ {filename_stem}.h5ad 中找不到 spatial 信息")
        except Exception as e:
            print(f"❌ 读取 h5ad 失败: {e}")
            
    return None

# ==========================================
# 修复版：自动计算 spot_size
# ==========================================
def process_fusion(filename, resolution=0.5):
    filename_stem = filename.replace('.pt', '')
    
    # 1. 加载特征
    emb_path = os.path.join(EMB_DATA_DIR, filename)
    if not os.path.exists(emb_path): return
    embeddings = torch.load(emb_path).numpy()
    
    # 2. 加载坐标
    coords = load_coords_from_original(filename_stem)
    if coords is None:
        print(f"❌ 跳过 {filename}: 无法在原始数据中找到坐标")
        return

    if embeddings.shape[0] != coords.shape[0]:
        print(f"⚠️ 维度不匹配 {filename}，跳过")
        return

    print(f"[-] 处理: {filename_stem}")
    
    adata = ad.AnnData(X=embeddings)
    adata.obsm['spatial'] = coords
    
    # ==========================================
    # 【关键新增】自动计算 spot_size
    # ==========================================
    # 1. 计算坐标在 X 和 Y 方向的跨度
    span_x = coords[:, 0].max() - coords[:, 0].min()
    span_y = coords[:, 1].max() - coords[:, 1].min()
    # 2. 取最大跨度
    max_span = max(span_x, span_y)
    # 3. 经验法则：设置点大小为最大跨度的 1.5% 左右
    # 你可以调整这个 0.015 的系数来整体控制点的大小
    auto_spot_size = max_span * 0.015 
    
    # 防止对于极小坐标点变得太小，设置一个最小值
    auto_spot_size = max(auto_spot_size, 10) 

    print(f"    -> 坐标跨度: {max_span:.1f}, 自动计算点大小: {auto_spot_size:.1f}")

    # ==========================================

    try:
        sc.pp.neighbors(adata, use_rep='X', n_neighbors=15, metric='cosine')
        sc.tl.leiden(adata, resolution=resolution, key_added='spatial_domain')
        
        plt.rcParams["figure.figsize"] = (6, 6)
        save_name = f"{filename_stem}_domains.png"
        
        sc.pl.spatial(
            adata, 
            color='spatial_domain', 
            title=filename_stem,
            spot_size=auto_spot_size,  # <--- 使用自动计算的值
            palette='tab20', 
            show=False
        )
        plt.savefig(os.path.join(RESULT_DIR, save_name), dpi=300, bbox_inches='tight')
        plt.close()
        adata.write(os.path.join(RESULT_DIR, f"{filename_stem}_result.h5ad"))
        
    except Exception as e:
        print(f"❌ 分析出错 {filename}: {e}")

if __name__ == "__main__":
    # 检查原始数据目录是否存在
    if not os.path.exists(RAW_DATA_DIR):
        print(f"❌ 错误: 找不到原始数据目录: {RAW_DATA_DIR}")
        print("请修改脚本中的 RAW_DATA_DIR 为存放 .h5ad 文件的真实路径！")
    else:
        file_list = sorted(glob.glob(os.path.join(EMB_DATA_DIR, "*.pt")))
        print(f"[-] 开始处理 {len(file_list)} 个样本...")
        
        for file_path in tqdm(file_list):
            filename = os.path.basename(file_path)
            process_fusion(filename, resolution=0.6)
            
        print(f"✅ 完成！结果在 {RESULT_DIR}")

[-] 开始处理 65 个样本...


  0%|          | 0/65 [00:00<?, ?it/s]

[-] 处理: 151673
    -> 坐标跨度: 9147.0, 自动计算点大小: 137.2


/data/home/wangzz_group/zhaipengyuan/.conda/envs/BEPH/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_3765830/1200640473.py:97: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=resolution, key_added='spatial_domain')
  2%|▏         | 1/65 [00:12<13:50, 12.98s/it]

[-] 处理: ColorectalCancer_10x
    -> 坐标跨度: 9239.0, 自动计算点大小: 138.6


...

[-] 处理: LungCancer_10x
    -> 坐标跨度: 24514.0, 自动计算点大小: 367.7


 98%|█████████▊| 64/65 [02:50<00:02,  2.40s/it]

[-] 处理: ProstateAcinarCancer_10x
    -> 坐标跨度: 19487.0, 自动计算点大小: 292.3


100%|██████████| 65/65 [02:55<00:00,  2.69s/it]

✅ 完成！结果在 spatial_domains_result


In [2]:
import torch
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import os
from sklearn.metrics import adjusted_rand_score

# =========================================================================
# 1. 配置区域 (请确认路径准确)
# =========================================================================

# 模型生成的特征 (.pt) 所在目录
EMB_DATA_DIR = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/kz_data/Graph_embeddings/"

# 原始数据 (.h5ad) 所在目录 (用于读取坐标)
RAW_DATA_DIR = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/kz_data/Raw_Data/Log1p/"

# 真实标签文件 (.txt) 的完整路径 (注意：文件名要对应好)
TRUTH_FILE_PATH = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/kz_data/151673/raw/151673_truth.txt"

# 要分析的文件名
TARGET_FILENAME = "151673.pt"

# 目标聚类数量 (DLPFC 通常设为 5 或 7)
TARGET_N_CLUSTERS = 7

# =========================================================================
# 2. 辅助函数定义
# =========================================================================

def load_coords_from_original(filename_stem):
    """从原始 h5ad 加载空间坐标"""
    h5ad_path = os.path.join(RAW_DATA_DIR, f"{filename_stem}.h5ad")
    if os.path.exists(h5ad_path):
        try:
            adata_orig = sc.read_h5ad(h5ad_path)
            if 'spatial' in adata_orig.obsm:
                return adata_orig.obsm['spatial']
            elif 'x' in adata_orig.obs and 'y' in adata_orig.obs:
                return np.column_stack((adata_orig.obs['x'], adata_orig.obs['y']))
        except Exception as e:
            print(f"❌ 读取 h5ad 失败: {e}")
    return None

def binary_search_leiden_resolution(adata, target_k, max_steps=20):
    """二分查找法：自动寻找最佳 resolution 以逼近 target_k"""
    print(f"[*] 正在运行 Leiden 聚类 (目标类别数: {target_k})...")
    
    low = 0.1
    high = 3.0
    res = 1.0
    best_res = res
    best_diff = float('inf')
    
    sc.pp.neighbors(adata, use_rep='X', n_neighbors=15, metric='cosine')

    for step in range(max_steps):
        sc.tl.leiden(adata, resolution=res, key_added='temp_domain')
        current_k = len(adata.obs['temp_domain'].unique())
        
        # 精确命中
        if current_k == target_k:
            adata.obs['spatial_domain'] = adata.obs['temp_domain']
            return
        
        # 记录最佳
        diff = abs(current_k - target_k)
        if diff < best_diff:
            best_diff = diff
            best_res = res
            adata.obs['spatial_domain'] = adata.obs['temp_domain']

        # 二分调整
        if current_k > target_k:
            high = res
        else:
            low = res
        res = (low + high) / 2
        
        if (high - low) < 0.01:
            break

    # 循环结束，使用最佳结果
    print(f"    -> 最终使用 resolution={best_res:.3f} (得到 {len(adata.obs['spatial_domain'].unique())} 类)")
    sc.tl.leiden(adata, resolution=best_res, key_added='spatial_domain')

def load_truth_and_calc_ari(adata, truth_path):
    """读取无表头 txt 并计算 ARI"""
    if not os.path.exists(truth_path):
        print(f"❌ 找不到真实标签文件: {truth_path}")
        return

    try:
        # 读取无表头 txt: 第一列是 Barcode，第二列是 Label
        truth_df = pd.read_csv(
            truth_path, 
            sep=None,       # 自动识别分隔符 (Tab 或空格)
            engine='python', 
            header=None,    # 无表头
            names=['barcode', 'ground_truth'] # 手动命名
        )
        truth_df.set_index('barcode', inplace=True)
        
        # 合并标签 (Left Join)
        if 'ground_truth' in adata.obs:
            adata.obs.drop(columns=['ground_truth'], inplace=True)
        adata.obs = adata.obs.join(truth_df, how='left')
        
        # 提取有效数据 (同时有预测值和真实值的点)
        valid_mask = (~pd.isnull(adata.obs['ground_truth'])) & (~pd.isnull(adata.obs['spatial_domain']))
        
        if valid_mask.sum() == 0:
            print("❌ 错误: Barcode 无法匹配 (请检查 .txt 里的 Barcode 是否带 '-1' 后缀)")
            return

        # 计算 ARI
        valid_truth = adata.obs.loc[valid_mask, 'ground_truth']
        valid_preds = adata.obs.loc[valid_mask, 'spatial_domain']
        score = adjusted_rand_score(valid_truth, valid_preds)
        
        print("\n" + "="*40)
        print(f"✅ 最终结果 (ARI Score)")
        print(f"   文件名: {TARGET_FILENAME}")
        print(f"   有效点数: {valid_mask.sum()}")
        print(f"   ARI: {score:.4f}")
        print("="*40 + "\n")

    except Exception as e:
        print(f"❌ 计算 ARI 出错: {e}")

# =========================================================================
# 3. 主程序
# =========================================================================

if __name__ == "__main__":
    filename_stem = TARGET_FILENAME.replace('.pt', '')
    emb_path = os.path.join(EMB_DATA_DIR, TARGET_FILENAME)
    
    # 1. 检查文件
    if not os.path.exists(emb_path):
        print(f"❌ 找不到特征文件: {emb_path}")
        exit()
        
    # 2. 加载数据
    print(f"[-] 加载特征: {TARGET_FILENAME}")
    embeddings = torch.load(emb_path).numpy()
    coords = load_coords_from_original(filename_stem)
    
    if coords is None:
        print("❌ 无法加载坐标，退出。")
        exit()
        
    if embeddings.shape[0] != coords.shape[0]:
        print(f"❌ 维度不匹配: 特征 {embeddings.shape[0]} vs 坐标 {coords.shape[0]}")
        exit()

    # 3. 构建 AnnData
    adata = ad.AnnData(X=embeddings)
    adata.obsm['spatial'] = coords
    adata.obs_names = sc.read_h5ad(os.path.join(RAW_DATA_DIR, f"{filename_stem}.h5ad")).obs_names

    # 4. 聚类
    binary_search_leiden_resolution(adata, target_k=TARGET_N_CLUSTERS)

    # 5. 计算 ARI
    load_truth_and_calc_ari(adata, TRUTH_FILE_PATH)

[-] 加载特征: 151673.pt
[*] 正在运行 Leiden 聚类 (目标类别数: 7)...

✅ 最终结果 (ARI Score)
   文件名: 151673.pt
   有效点数: 3611
   ARI: 0.2673

